<a href="https://colab.research.google.com/github/A-ros1076/BUS118s/blob/Dev/Exercise_2_Code_Generation_with_ReACT_Prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2: Code Generation with ReACT Prompting

**Goal:** Use a ReACT-style approach (Reason → Act → Observe → Reflect) to generate Python code and execute it.

| Stage | What happens |
|-------|--------------|
| Reason | Gemini plans the approach before writing any code |
| Act | Gemini generates the Python code based on its plan |
| Observe | Code is executed in Colab and output is captured |
| Reflect | Gemini reviews the output and confirms it meets requirements |

**Task:** Generate a Python calculator that supports addition, subtraction, multiplication, and division — with input validation and error handling.

**Tool:** Google Colab + Gemini API (`gemini-2.5-flash`)

In [1]:
!pip install google-generativeai -q

In [2]:
import google.generativeai as genai
from google.colab import userdata
import re

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

def ask_gemini(prompt: str) -> str:
    """
    Sends a prompt to Gemini and returns the text response.

    Args:
        prompt: The full prompt string to send.

    Returns:
        Gemini's response as a stripped string.
    """
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        print(f"An error occurred: {e}")
        return "Request failed."

print("Gemini client ready.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini client ready.


## ReACT Cycle

All four stages run in sequence. Gemini's reasoning from Stage 1 feeds into Stage 2 (code generation). The executed output from Stage 3 feeds into Stage 4 (reflection).

In [3]:
# Test cases to run against the generated calculator
TEST_CASES = [
    (10, '+', 5),
    (10, '-', 3),
    (6,  '*', 7),
    (10, '/', 4),
    (5,  '/', 0),    # division by zero
    (9,  '%', 3),    # invalid operator
    ('a', '+', 3),   # non-numeric input
]

# ============================================================
# STAGE 1: REASON — Gemini plans before writing any code
# ============================================================
print("=" * 60)
print("STAGE 1: REASON — Gemini plans the approach")
print("=" * 60)

reason_prompt = """
You are a Python engineer using the ReACT framework.

REASON stage: Before writing any code, plan your approach in plain English.

Task: Build a Python calculator function that:
- Supports addition, subtraction, multiplication, and division
- Validates that the operator is one of the four supported symbols
- Handles division by zero with a clear error message
- Rejects non-numeric inputs with a clear error message
- Returns results rounded to 2 decimal places

List your step-by-step plan and the error cases you will handle.
Do NOT write any code yet — reasoning only.
"""

reason_output = ask_gemini(reason_prompt)
print(reason_output)

# ============================================================
# STAGE 2: ACT — Gemini writes the code based on its plan
# ============================================================
print("\n" + "=" * 60)
print("STAGE 2: ACT — Gemini generates the code")
print("=" * 60)

act_prompt = f"""
You are a Python engineer using the ReACT framework.

ACT stage: You already planned this approach:
{reason_output}

Now implement it. Write a function called calculate(num1, operator, num2).

Requirements:
- Supports: +, -, *, /
- Raises ValueError for unsupported operators
- Raises ZeroDivisionError for division by zero
- Raises TypeError for non-numeric inputs
- Returns result as a float rounded to 2 decimal places
- Include a docstring listing Args, Returns, and Raises

Return ONLY the raw Python code — no explanation, no markdown fences.
"""

act_output = ask_gemini(act_prompt)
code = re.sub(r'```python\n?|```', '', act_output).strip()
print(code)

# ============================================================
# STAGE 3: OBSERVE — Execute the code and run all test cases
# ============================================================
print("\n" + "=" * 60)
print("STAGE 3: OBSERVE — Executing the generated code")
print("=" * 60)

exec(code, globals())

for num1, op, num2 in TEST_CASES:
    try:
        result = calculate(num1, op, num2)
        print(f"{num1} {op} {num2} = {result}")
    except (ValueError, ZeroDivisionError, TypeError) as e:
        print(f"Error caught: {e}")

# ============================================================
# STAGE 4: REFLECT — Gemini reviews the output against requirements
# ============================================================
print("\n" + "=" * 60)
print("STAGE 4: REFLECT — Gemini reviews the output")
print("=" * 60)

reflect_prompt = f"""
You are a Python engineer using the ReACT framework.

REFLECT stage: Review the generated code and confirm it meets all requirements.

Requirements:
- Supports +, -, *, /
- Raises ValueError for unsupported operators
- Raises ZeroDivisionError for division by zero
- Raises TypeError for non-numeric inputs
- Returns results rounded to 2 decimal places
- Includes a docstring

Generated code:
{code}

In 3-5 sentences, confirm whether the code meets each requirement and suggest
one improvement for a future iteration.
"""

reflect_output = ask_gemini(reflect_prompt)
print(reflect_output)

STAGE 1: REASON — Gemini plans the approach
REASON:

The task requires building a Python calculator function with specific functionalities and robust error handling. I will break down the development into logical steps, prioritizing input validation and error handling before performing any calculations.

**Step-by-step plan:**

1.  **Function Definition:**
    *   Define a Python function, let's call it `calculator`, that accepts three arguments: `num1`, `operator`, and `num2`.

2.  **Validate Operator:**
    *   First, check if the `operator` argument is one of the allowed symbols: `'+'`, `'-'`, `'*'`, or `'/'`.
    *   If the operator is not valid, immediately return a specific error message string.

3.  **Validate Operands (Convert to Numeric Type):**
    *   Attempt to convert both `num1` and `num2` to `float` types. This handles both integer and decimal inputs.
    *   Use a `try-except ValueError` block for this conversion. If either conversion fails, it means one or both inputs 